In [2]:
import pandas as pd

data = {
    "CustomerID":[
        101,101,101,
        102,102,
        103,103,103,103,
        104,104
    ],
    
    "OrderID":[
        1,2,3,
        4,5,
        6,7,8,9,
        10,11
    ],
    
    "OrderDate":[
        "2023-01-01","2023-02-10","2023-03-15",
        "2023-01-05","2023-02-20",
        "2023-01-02","2023-01-25","2023-02-18","2023-03-10",
        "2023-02-01","2023-03-01"
    ],
    
    "Amount":[
        200,150,300,
        400,250,
        100,120,130,200,
        500,350
    ]
}

df = pd.DataFrame(data)

df["OrderDate"] = pd.to_datetime(df["OrderDate"])

print(df)


    CustomerID  OrderID  OrderDate  Amount
0          101        1 2023-01-01     200
1          101        2 2023-02-10     150
2          101        3 2023-03-15     300
3          102        4 2023-01-05     400
4          102        5 2023-02-20     250
5          103        6 2023-01-02     100
6          103        7 2023-01-25     120
7          103        8 2023-02-18     130
8          103        9 2023-03-10     200
9          104       10 2023-02-01     500
10         104       11 2023-03-01     350


In [16]:
# Problem 1 — First Purchase Per Customer
# Find the first order each customer placed.
# Expected output columns: | CustomerID | OrderID | OrderDate | Amount |

first_orders = df.sort_values(by="OrderDate").groupby("CustomerID").first().reset_index()
print(first_orders[["CustomerID", "OrderID", "OrderDate", "Amount"]])

   CustomerID  OrderID  OrderDate  Amount
0         101        1 2023-01-01     200
1         102        4 2023-01-05     400
2         103        6 2023-01-02     100
3         104       10 2023-02-01     500


In [17]:
# Problem 2 — Total Revenue Per Customer
# Create a dataframe showing: | CustomerID | Total_Revenue |
total_revenue = (df.groupby("CustomerID")["Amount"].sum().reset_index(name="Total_Revenue").sort_values("Total_Revenue", ascending=False))
print(total_revenue)

   CustomerID  Total_Revenue
3         104            850
0         101            650
1         102            650
2         103            550


In [19]:
# Problem 3 — Repeat Customers
# Hint:groupby + count , Then filter customers with more than 1 order.
repeat_customers = df.groupby("CustomerID")["OrderID"].count().reset_index(name="Total_Orders").query("Total_Orders > 1")
print(repeat_customers)

   CustomerID  Total_Orders
0         101             3
1         102             2
2         103             4
3         104             2


In [20]:
# Find customers whose latest order amount is higher than their first order amount.
#use groupby first() last()
first_last = (df.sort_values("OrderDate").groupby("CustomerID")["Amount"].agg(First_Amount="first", Last_Amount="last").reset_index())

growth_customers = first_last[first_last["Last_Amount"] > first_last["First_Amount"]]
print(growth_customers[["CustomerID", "First_Amount", "Last_Amount"]])

   CustomerID  First_Amount  Last_Amount
0         101           200          300
2         103           100          200


In [30]:
# Top 2 highest Amount orders per customer use groupby & nlargest
top_orders = df.groupby("CustomerID", group_keys=False).apply(lambda x: x.nlargest(2, "Amount"), include_groups=False).reset_index(drop=True)
print(top_orders[["OrderID", "OrderDate", "Amount"]])

   OrderID  OrderDate  Amount
0        3 2023-03-15     300
1        1 2023-01-01     200
2        4 2023-01-05     400
3        5 2023-02-20     250
4        9 2023-03-10     200
5        8 2023-02-18     130
6       10 2023-02-01     500
7       11 2023-03-01     350


In [22]:
top_orders = (df.sort_values(["CustomerID","Amount"], ascending=[True, False]).groupby("CustomerID").head(2))
print(top_orders[["CustomerID", "OrderID", "OrderDate", "Amount"]])

    CustomerID  OrderID  OrderDate  Amount
2          101        3 2023-03-15     300
0          101        1 2023-01-01     200
3          102        4 2023-01-05     400
4          102        5 2023-02-20     250
8          103        9 2023-03-10     200
7          103        8 2023-02-18     130
9          104       10 2023-02-01     500
10         104       11 2023-03-01     350


In [26]:
# Days since previous order
df["Days_Since_Prev"] = df.sort_values("OrderDate").groupby("CustomerID")["OrderDate"].diff().dt.days
print(df[["CustomerID", "OrderID", "OrderDate", "Amount", "Days_Since_Prev"]])

    CustomerID  OrderID  OrderDate  Amount  Days_Since_Prev
0          101        1 2023-01-01     200              NaN
1          101        2 2023-02-10     150             40.0
2          101        3 2023-03-15     300             33.0
3          102        4 2023-01-05     400              NaN
4          102        5 2023-02-20     250             46.0
5          103        6 2023-01-02     100              NaN
6          103        7 2023-01-25     120             23.0
7          103        8 2023-02-18     130             24.0
8          103        9 2023-03-10     200             20.0
9          104       10 2023-02-01     500              NaN
10         104       11 2023-03-01     350             28.0


In [41]:
# Find customers who made purchases in consecutive months
df = df.sort_values(["CustomerID", "OrderDate"])

df["Month"] = df["OrderDate"].dt.month

df["Month_Diff"] = df.groupby("CustomerID")["Month"].diff()

consecutive = df[df["Month_Diff"] == 1]

print(consecutive[["CustomerID", "OrderDate", "Month"]])

customers = consecutive["CustomerID"].unique()
print(customers)

    CustomerID  OrderDate  Month
1          101 2023-02-10      2
2          101 2023-03-15      3
4          102 2023-02-20      2
7          103 2023-02-18      2
8          103 2023-03-10      3
10         104 2023-03-01      3
[101 102 103 104]


In [47]:
df["Return_days"] = df.sort_values("OrderDate").groupby("CustomerID")["OrderDate"].diff().dt.days
Return_in_30_days = df[df["Return_days"].le(30) & df["Return_days"].notna()]
print(Return_in_30_days[["CustomerID", "OrderID", "OrderDate", "Amount", "Return_days"]])

retention = (
    df[df["Return_days"] <= 30]
    .groupby("CustomerID")
    .size()
    .reset_index(name="Returned_in_30_days")
)
print(retention)

    CustomerID  OrderID  OrderDate  Amount  Return_days
6          103        7 2023-01-25     120         23.0
7          103        8 2023-02-18     130         24.0
8          103        9 2023-03-10     200         20.0
10         104       11 2023-03-01     350         28.0
   CustomerID  Returned_in_30_days
0         103                    3
1         104                    1


In [14]:
customer_metrics = (
    df.groupby("CustomerID")["Amount"]
      .agg(
          orders_count="count",
          total_revenue="sum",
          avg_order_value="mean"
      )
      .reset_index()
)

print(customer_metrics)

   CustomerID  orders_count  total_revenue  avg_order_value
0         101             3            650       216.666667
1         102             2            650       325.000000
2         103             4            550       137.500000
3         104             2            850       425.000000
